# Part 1: Dataset Setup, Projection, and Spatial Alignment

This notebook covers the initial steps of the AHP Beehive Suitability Workflow:
1. **Environment Setup & Imports**
2. **Loading Configuration and Paths**
3. **Reprojection & Alignment** of all 10 criterion rasters using the LULC layer as the spatial template
4. **Study Area Mask Extraction** (Rasterizing Shapefile district boundary & intersecting with valid data cells)
5. **Data Range Profiling** (Pre-processing stats)

In [ ]:
# 1. Environment Imports
import os
import sys
import numpy as np
import rasterio
import warnings
warnings.filterwarnings('ignore')

# Add root directory to path to import src modules if running on Colab or locally
sys.path.append(os.path.abspath('..'))

### Google Colab Mount (Optional)
If you are running this workflow on Google Colab, uncomment and run the cell below to mount your drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

### 2. Loading configuration settings
We load file paths, names, and parameters defined in `src.config`.

In [ ]:
from src.config import unclassified_paths, classified_paths, shapefile_path

print("Unclassified raster paths:")
for k, v in unclassified_paths.items():
    print(f" - {k}: {v}")

print("\nClassified raster paths:")
for k, v in classified_paths.items():
    print(f" - {k}: {v}")

### 3. Load & Spatial Alignment
Align all criteria rasters so they share the exact dimensions, cell size, and projection coordinates as the LULC template.

In [ ]:
from src.alignment import load_aligned

template_path = classified_paths['LULC']
data_stack = {}
meta = None

print("Loading & aligning rasters...")
for name, path in {**unclassified_paths, **classified_paths}.items():
    if path == template_path:
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan
            data_stack[name] = arr
            meta = src.meta.copy()
    else:
        data_stack[name] = load_aligned(template_path, path)
    print(f"  ✓ Aligned: {name}")

### 4. Extracting and Intersecting Study Area Mask
We clip the study area boundary mask using Kavre's Shapefile district boundary intersected with the valid raster extent.

In [ ]:
from src.alignment import create_study_area_mask

study_area_mask = create_study_area_mask(shapefile_path, meta, data_stack)
print(f"Study area boundary mask resolved with {study_area_mask.sum():,} valid pixels.")

### 5. Pre-processing Ranges and Stats profiling

In [ ]:
# Pre-processing ranges cleaning
for flag in [-9999, -9998, 9999, 32767, 65535, -32768, 32768]:
    for name in data_stack:
        data_stack[name][data_stack[name] == flag] = np.nan

# Forest Cover adjustments (0-1 proportion to 0-100% scale)
data_stack['Forest_Cover'] = data_stack['Forest_Cover'] * 100.0

# NDVI clip to actual boundaries
data_stack['NDVI'] = np.clip(data_stack['NDVI'], -0.3357, 0.8905)

# Print out RAW ranges stats
raw_stats = {}
print("RAW DATA RANGES:")
for name, arr in data_stack.items():
    v = arr[~np.isnan(arr)]
    if len(v) > 0:
        raw_stats[name] = {
            'min': float(v.min()),
            'max': float(v.max()),
            'mean': float(v.mean())
        }
        print(f"  * {name:15s} -> min: {v.min():10.4f} | max: {v.max():10.4f} | mean: {v.mean():10.4f}")